In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/medquad.csv')
df.head()  # Preview first few rows


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


# Drop rows with missing QA

In [3]:
df = df.dropna(subset=['question', 'answer'])


# Restructure for Fine-Tuning Format

In [4]:
processed_data = []

for _, row in df.iterrows():
    question = row['question'].strip()
    answer = row['answer'].strip()

    processed_data.append({
        "instruction": question,
        "input": "",
        "output": answer
    })


In [6]:
import json

with open('/content/drive/MyDrive/medquad_preprocessed.json', 'w') as f:
    json.dump(processed_data, f, indent=2)


In [7]:
#check the preprocess data
import json

# Load the preprocessed file
with open('/content/drive/MyDrive/medquad_preprocessed.json', 'r') as f:
    processed_data = json.load(f)

# Print the number of records
print(f"Total examples: {len(processed_data)}")

# View first example
print(json.dumps(processed_data[0], indent=2))


Total examples: 16407
{
  "instruction": "What is (are) Glaucoma ?",
  "input": "",
  "output": "Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts

In [8]:
!pip install -q transformers accelerate bitsandbytes peft datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [10]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"


In [11]:
from huggingface_hub import login
login()


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [13]:
!pip install peft bitsandbytes accelerate

# Set up quantization config using BitsAndBytesConfig

In [14]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


# Reload your model using that quant config

In [15]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# Prepare model for k-bit training (LoRA setup)

In [16]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)


# Apply LoRA using PEFT

In [17]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # These are common for Mistral/LLM models
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"  # because you're using it for text generation
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # just to confirm it's set up correctly


trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


# Set up training arguments and Trainer:

In [18]:
from transformers import TrainingArguments

output_dir = "./mistral-lora-finetuned"

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    output_dir=output_dir,
    save_steps=500,
    save_total_limit=2,
    report_to=[]  # disables logging to W&B or HF
)


#  Load JSON and Create Custom Dataset

In [19]:
import json

# Load your JSON data
with open("/content/drive/MyDrive/medquad_preprocessed.json", "r") as f:
    data = json.load(f)

# Define your custom dataset class
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Combine instruction, input (if any), and output
        prompt = item["instruction"]
        if item["input"]:
            prompt += "\n" + item["input"]
        full_text = f"### Instruction:\n{prompt}\n\n### Response:\n{item['output']}"

        # Tokenize
        tokenized = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        tokenized["labels"] = tokenized["input_ids"].clone()
        return {key: val.squeeze(0) for key, val in tokenized.items()}


# Instantiate the dataset

In [20]:
# Instantiate the dataset
train_dataset = InstructionDataset(data, tokenizer)


# Create a DataLoader

In [21]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)


# Train the model

In [22]:
# Step 1: Load the tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
tokenizer.pad_token = tokenizer.eos_token  # To avoid the padding error

# Step 2: Load JSON data
import json

with open("/content/drive/MyDrive/medquad_preprocessed.json", "r") as f:
    data = json.load(f)

# Step 3: Create a custom dataset
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = item["instruction"]
        if item["input"]:
            prompt += "\n" + item["input"]
        full_text = f"### Instruction:\n{prompt}\n\n### Response:\n{item['output']}"

        tokenized = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        tokenized["labels"] = tokenized["input_ids"].clone()
        return {key: val.squeeze(0) for key, val in tokenized.items()}

# Step 4: Instantiate the dataset (shortened for faster testing)
train_dataset = InstructionDataset(data[:100], tokenizer)

# Step 5: Setup training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,  # Run for only 100 steps
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    output_dir="./mistral-lora-finetuned",
    fp16=True,
    save_strategy="no",
    logging_strategy="steps",
    remove_unused_columns=False,
    report_to="none"  # Disable Weights & Biases
)

# Step 6: Create the Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,  # your LoRA-merged model
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

# Step 7: Train!
trainer.train()


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

<ipython-input-22-cefb597f0ea8>:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args,

Step,Training Loss
10,6.028700
20,0.967900
30,0.748100
40,0.647300
50,0.616200
60,0.506500
70,0.563000
80,0.464700
90,0.434900
100,0.412600


TrainOutput(global_step=100, training_loss=1.1389879751205445, metrics={'train_runtime': 1605.7514, 'train_samples_per_second': 0.498, 'train_steps_per_second': 0.062, 'total_flos': 1.6871609767821312e+16, 'train_loss': 1.1389879751205445, 'epoch': 7.72})

In [23]:
from google.colab import drive
drive.mount('/content/drive')

# Save model in Drive
trainer.save_model("/content/drive/MyDrive/mistral-lora-finetuned-final")
tokenizer.save_pretrained("/content/drive/MyDrive/mistral-lora-finetuned-final")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/mistral-lora-finetuned-final/tokenizer_config.json',
 '/content/drive/MyDrive/mistral-lora-finetuned-final/special_tokens_map.json',
 '/content/drive/MyDrive/mistral-lora-finetuned-final/tokenizer.model',
 '/content/drive/MyDrive/mistral-lora-finetuned-final/added_tokens.json',
 '/content/drive/MyDrive/mistral-lora-finetuned-final/tokenizer.json')

In [ ]:
trainer.save_model("./mistral-lora-finetuned-final")
tokenizer.save_pretrained("./mistral-lora-finetuned-final")


('./mistral-lora-finetuned-final/tokenizer_config.json',
 './mistral-lora-finetuned-final/special_tokens_map.json',
 './mistral-lora-finetuned-final/tokenizer.model',
 './mistral-lora-finetuned-final/added_tokens.json',
 './mistral-lora-finetuned-final/tokenizer.json')

In [ ]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.8 MB/s eta 0:00:00
